

## KOL Booking Data Cleaning Summary

This notebook prepares KOL booking data for **post-clustering marketing analysis** by cleaning raw operational records and engineering restaurant-month marketing signals. 
Output: Saved the cleaned and aggregated dataset to kol_cleaned_booking_monthly.parquet

### What was done

* **Loaded & combined datasets**

  * Merged Singapore and Thailand KOL booking files into a single table
  * Added a `market` column to distinguish regions

* **Column standardisation & filtering**

  * Standardised column names across source files
  * Retained only analytically relevant columns (e.g. `KOL_ID`, `Restaurant ID`, `Followers`, `Package Type`, `Target Date`, `Sales Check`)

* **Data quality checks**

  * Removed test records, invalid entries, and rows without valid `Restaurant ID`
  * Filtered to keep only executed campaigns (`Booked`, `Validated`)

* **Data type cleaning**

  * Converted `Restaurant ID` to numeric format for reliable joins
  * Cleaned `Followers` by converting mixed formats (commas, K/M shorthand) into numeric follower counts
  * Parsed `Target Date` into datetime format using day-month-year convention

* **Feature engineering**

  * Created `year_month` to align KOL activity with monthly momentum and clustering outputs
  * Engineered marketing indicator columns:

    * `is_kol_campaign` — flags presence of KOL activity in a given month
    * `is_high_reach_kol` — identifies campaigns involving high-reach influencers (≥100k followers)

* **Aggregation**

  * Aggregated data to a **restaurant–month** level
  * Generated summary marketing features:

    * `kol_campaigns` — number of KOL campaigns in the month
    * `unique_kols` — number of distinct influencers used
    * `avg_followers` — average follower count of KOLs used
    * `high_reach_campaigns` — number of high-reach KOL campaigns




1. load n combine datasets

We first standardise column names and add a market label so Singapore and Thailand data can be analysed together while still allowing market-specific segmentation later. Concatenating early simplifies downstream cleaning and ensures consistent logic across regions.

In [11]:
import pandas as pd
import numpy as np

sg = pd.read_csv("../data/kol/KOL_Booking SG.csv")
th = pd.read_csv("../data/kol/KOL_Booking TH.csv")

def standardise(df, market):
    df = df.copy()
    df.columns = df.columns.str.strip()
    df["market"] = market
    return df

sg = standardise(sg, "SG")
th = standardise(th, "TH")

kol_booking_raw = pd.concat([sg, th], ignore_index=True)


2. keep only valid / useful columns

In [12]:
KEEP_COLS = [
    "KOL_ID",
    "Followers",
    "Country",
    "Restaurant ID",
    "Restaurant name",
    "Cuisine Type",
    "Package Type",
    "Package price",
    "Target Date",
    "Sales Check",
    "Booking ID",
    "market"
]

kol_booking = kol_booking_raw[KEEP_COLS]


3. remove invalid rows

In [13]:
kol_booking = kol_booking[
    kol_booking["Restaurant ID"].notna() &
    kol_booking["KOL_ID"].notna() &
    ~kol_booking["KOL_ID"].isin(["#REF!", "HH-TESTING"])
]


4. fix data types

In [14]:
# Restaurant ID
kol_booking["Restaurant ID"] = (
    pd.to_numeric(kol_booking["Restaurant ID"], errors="coerce")
    .astype("Int64")
)
kol_booking = kol_booking[kol_booking["Restaurant ID"].notna()]

# Followers
def clean_followers(x):
    x = str(x).strip().upper()
    if x.endswith("K"):
        return float(x.replace("K", "")) * 1_000
    elif x.endswith("M"):
        return float(x.replace("M", "")) * 1_000_000
    else:
        return float(x.replace(",", ""))

kol_booking["Followers"] = kol_booking["Followers"].apply(clean_followers)
kol_booking.loc[kol_booking["Followers"] == 0, "Followers"] = np.nan

# Target Date
kol_booking["Target Date"] = pd.to_datetime(
    kol_booking["Target Date"],
    dayfirst=True,
    errors="coerce"
)



In [15]:
kol_booking["Followers"].describe()


count       511.000000
mean      65482.013699
std       86330.767012
min        2338.000000
25%       19148.500000
50%       35936.000000
75%       78525.000000
max      512171.000000
Name: Followers, dtype: float64

5. keep only executed campaigns

In [16]:
kol_booking = kol_booking[
    kol_booking["Sales Check"].isin(["Booked", "Validated"])
]


6. feature engineering 

- is_kol_campaign: Marks that a restaurant had any KOL marketing activity in that month, so we can easily compare months with vs without influencer campaigns.

- is_high_reach_kol: Flags whether the campaign involved a high-reach influencer (≥100k followers), allowing us to compare the impact of large KOLs vs micro-KOLs later on

In [17]:
kol_booking["year_month"] = (
    kol_booking["Target Date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

kol_booking["is_kol_campaign"] = 1
kol_booking["is_high_reach_kol"] = (kol_booking["Followers"] >= 100_000).astype(int)


7. Aggregrate to per restaurant per month

- Combines multiple KOL campaigns in the same month into one clean row per restaurant per month, matching the granularity of momentum and clustering outputs.

- kol_campaigns: total number of KOL campaigns run that month (marketing intensity)
- unique_kols: number of different influencers used (creator diversity)
- avg_followers: average audience size of KOLs used (typical reach)
- high_reach_campaigns: number of campaigns using large KOLs (big-influencer exposure)

In [18]:
kol_booking_monthly = (
    kol_booking
    .groupby(["Restaurant ID", "year_month"], as_index=False)
    .agg(
        kol_campaigns=("KOL_ID", "count"),
        unique_kols=("KOL_ID", "nunique"),
        avg_followers=("Followers", "mean"),
        high_reach_campaigns=("is_high_reach_kol", "sum")
    )
    .rename(columns={"Restaurant ID": "restaurant_id"})
)

kol_booking_monthly.head()


,restaurant_id,year_month,kol_campaigns,unique_kols,avg_followers,high_reach_campaigns
0,33,2025-10-01,1,1,36300.0,0
1,280,2025-05-01,1,1,20381.0,0
2,280,2025-06-01,1,1,110178.0,1
3,425,2025-10-01,1,1,39700.0,0
4,425,2025-11-01,1,1,12000.0,0


In [19]:
kol_booking_monthly.to_parquet("kol_cleaned_booking_monthly.parquet", index=False)
